# OSRM and Fake GPS API Walkthrough

Run the setup cell first, then run the sections independently to inspect inputs, outputs, and behavior. The default examples use mocked OSRM data and a temporary route cache, so they require no network call and do not change the repository's routes.json.

Install the project requirements before running. To try a real OSRM server, set OSRM_BASE_URL in the notebook kernel and run the optional live-request cell. Truck dimensions are metadata unless the configured OSRM profile enforces them; standard OSRM geometry may omit altitude.

## 1. Imports and project setup

In [1]:
from pathlib import Path
from pprint import pprint
from tempfile import TemporaryDirectory
from unittest.mock import patch
import asyncio
import os
import sys

project_root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "app" / "main.py").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app import main
from fastapi.testclient import TestClient

print("Project:", project_root)
print("FastAPI app:", main.app.title)

Project: C:\Users\anura\Documents\SupplyNet-Testing_server
FastAPI app: Logistics Route Optimization Agent


## 2. Request models and validation
RouteRequest validates coordinate ranges and optional vehicle attributes before making a route request.

In [2]:
route_input = {
    "origin": {"lat": 30.7333, "lon": 76.7794},
    "destination": {"lat": 30.8333, "lon": 76.8794},
    "vehicle": {
        "truck_type": "HCV", "gvw_kg": 18000,
        "axle_count": 6, "height_m": 4.2, "width_m": 2.5,
    },
}
route_request = main.RouteRequest.model_validate(route_input)
print("Input:")
pprint(route_input)
print("Validated model:")
pprint(route_request.model_dump())

Input:
{'destination': {'lat': 30.8333, 'lon': 76.8794},
 'origin': {'lat': 30.7333, 'lon': 76.7794},
 'vehicle': {'axle_count': 6,
             'gvw_kg': 18000,
             'height_m': 4.2,
             'truck_type': 'HCV',
             'width_m': 2.5}}
Validated model:
{'destination': {'lat': 30.8333, 'lon': 76.8794},
 'origin': {'lat': 30.7333, 'lon': 76.7794},
 'vehicle': {'axle_count': 6,
             'gvw_kg': 18000.0,
             'height_m': 4.2,
             'truck_type': 'HCV',
             'width_m': 2.5}}


In [3]:
invalid_input = {
    "origin": {"lat": 120, "lon": 76},
    "destination": {"lat": 28, "lon": 77},
}
try:
    main.RouteRequest.model_validate(invalid_input)
except Exception as error:
    print(type(error).__name__)
    print(error)

ValidationError
1 validation error for RouteRequest
origin.lat
  Input should be less than or equal to 90 [type=less_than_equal, input_value=120, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/less_than_equal


## 3. Route distance helper
_route_distance_km returns great-circle distance between adjacent route points; the simulator uses it to measure progress along geometry segments.

In [4]:
point_a = main.RoutePoint(lat=30.7333, lon=76.7794, altitude_m=350)
point_b = main.RoutePoint(lat=30.8333, lon=76.8794, altitude_m=320)
distance_km = main._route_distance_km(point_a, point_b)
print("Point A:", point_a.model_dump())
print("Point B:", point_b.model_dump())
print(f"Distance: {distance_km:.3f} km")

Point A: {'lat': 30.7333, 'lon': 76.7794, 'altitude_m': 350.0}
Point B: {'lat': 30.8333, 'lon': 76.8794, 'altitude_m': 320.0}
Distance: 14.659 km


## 4. RouteStore: save, load, get, list
The cache is temporary for this example. Constructing a second RouteStore demonstrates loading the saved JSON from disk.

In [5]:
with TemporaryDirectory() as temporary_directory:
    cache_path = Path(temporary_directory) / "routes.json"
    store = main.RouteStore(cache_path)
    saved_route = main.SavedRoadRoute(
        route_id="demo-route-1", alternative_index=0,
        created_at=main.datetime.now(main.timezone.utc),
        origin=route_request.origin, destination=route_request.destination,
        vehicle=route_request.vehicle, distance_km=12.5,
        duration_seconds=1080, geometry=[point_a, point_b],
        routing_profile="driving",
    )
    store.add_many([saved_route])
    loaded_store = main.RouteStore(cache_path)
    print("File exists:", cache_path.exists())
    print("list_all IDs:", [route.route_id for route in loaded_store.list_all()])
    print("get result:", loaded_store.get("demo-route-1").model_dump(mode="json"))

File exists: True
list_all IDs: ['demo-route-1']
get result: {'route_id': 'demo-route-1', 'alternative_index': 0, 'created_at': '2026-10-01T10:38:16.707859Z', 'source': 'osrm', 'origin': {'lat': 30.7333, 'lon': 76.7794}, 'destination': {'lat': 30.8333, 'lon': 76.8794}, 'vehicle': {'truck_type': 'HCV', 'gvw_kg': 18000.0, 'axle_count': 6, 'height_m': 4.2, 'width_m': 2.5}, 'distance_km': 12.5, 'duration_seconds': 1080.0, 'geometry': [{'lat': 30.7333, 'lon': 76.7794, 'altitude_m': 350.0}, {'lat': 30.8333, 'lon': 76.8794, 'altitude_m': 320.0}], 'routing_profile': 'driving', 'profile_specific': False, 'vehicle_constraints_verified': False, 'warnings': []}


## 5. OSRMClient.get_routes with a mocked response
This checks the OSRM URL and query options, GeoJSON coordinate order, and meters-to-kilometers conversion without making a network call.
The private _advance_position helper updates the simulated coordinate only; advance() also updates elapsed time and appends GPS history.
The private _advance_position helper updates the simulated coordinate only; advance() also updates elapsed time and appends GPS history.

In [6]:
class MockOSRMResponse:
    def raise_for_status(self):
        return None

    def json(self):
        return {
            "code": "Ok",
            "routes": [{
                "distance": 12500,
                "duration": 1080,
                "geometry": {
                    "coordinates": [[76.7794, 30.7333], [76.8794, 30.8333]]
                },
            }],
        }


class MockAsyncClient:
    def __init__(self, **kwargs):
        self.timeout = kwargs.get("timeout")

    async def __aenter__(self):
        return self

    async def __aexit__(self, *_args):
        return False

    async def get(self, url, params):
        print("OSRM URL:", url)
        print("OSRM params:", params)
        return MockOSRMResponse()


with patch.object(main.httpx, "AsyncClient", MockAsyncClient):
    osrm_routes = await main.OSRMClient().get_routes(route_request)

print("Parsed route:")
pprint({
    **osrm_routes[0],
    "geometry": [point.model_dump() for point in osrm_routes[0]["geometry"]],
})

OSRM URL: http://127.0.0.1:5000/route/v1/driving/76.7794,30.7333;76.8794,30.8333
OSRM params: {'alternatives': 'true', 'overview': 'full', 'geometries': 'geojson', 'steps': 'false'}
Parsed route:
{'distance_km': 12.5,
 'duration_seconds': 1080.0,
 'geometry': [{'altitude_m': None, 'lat': 30.7333, 'lon': 76.7794},
              {'altitude_m': None, 'lat': 30.8333, 'lon': 76.8794}],
 'profile_specific': False,
 'routing_profile': 'HCV'}


## 6. GPSSimulation: initialization, interpolation, advance, snapshot
The simulator calculates distance traveled from speed and elapsed seconds, interpolates coordinates and altitude, and appends timestamped GPS history.

In [7]:
demo_route = main.SavedRoadRoute(
    route_id="gps-demo", alternative_index=0,
    created_at=main.datetime.now(main.timezone.utc),
    origin=route_request.origin, destination=route_request.destination,
    distance_km=12.5, duration_seconds=1080,
    geometry=[point_a, point_b], routing_profile="driving",
)
simulation_request = main.StartSimulationRequest(
    vehicle_id="TRUCK-DEMO", route_id="gps-demo",
    speed_kmh=45, interval_seconds=30, auto_start=False,
)
simulation = main.GPSSimulation(simulation_request, demo_route)
print("Initial history:", simulation.history[0].model_dump(mode="json"))
simulation._advance_position(120)
print("Position after _advance_position(120):", simulation.current.model_dump())
print("snapshot():", simulation.snapshot().model_dump(mode="json"))

Initial history: {'lat': 30.7333, 'lon': 76.7794, 'altitude_m': 350.0, 'speed_kmh': 45.0, 'timestamp': '2026-10-01T10:38:16.718399Z'}
Position after _advance_position(120): {'lat': 30.743532277323233, 'lon': 76.78963227732324, 'altitude_m': 346.9303168030302}
snapshot(): {'lat': 30.743532277323233, 'lon': 76.78963227732324, 'altitude_m': 346.9303168030302, 'speed_kmh': 45.0, 'timestamp': '2026-10-01T10:38:16.718399Z', 'vehicle_id': 'TRUCK-DEMO', 'route_id': 'gps-demo', 'status': 'PAUSED', 'route_progress_percent': 10.2323}


In [8]:
before = simulation.snapshot()
await simulation.advance(30)
after = simulation.snapshot()
print("Before:", before.model_dump(mode="json"))
print("After advance(30):", after.model_dump(mode="json"))
print("History records:", len(simulation.history))

Before: {'lat': 30.743532277323233, 'lon': 76.78963227732324, 'altitude_m': 346.9303168030302, 'speed_kmh': 45.0, 'timestamp': '2026-10-01T10:38:16.718399Z', 'vehicle_id': 'TRUCK-DEMO', 'route_id': 'gps-demo', 'status': 'PAUSED', 'route_progress_percent': 10.2323}
After advance(30): {'lat': 30.74609034665404, 'lon': 76.79219034665404, 'altitude_m': 346.16289600378775, 'speed_kmh': 45.0, 'timestamp': '2026-10-01T10:38:46.718399Z', 'vehicle_id': 'TRUCK-DEMO', 'route_id': 'gps-demo', 'status': 'PAUSED', 'route_progress_percent': 12.7903}
History records: 2


## 7. Realtime task and cancellation
This runs the background worker for one one-second interval, then cancels it. Normal API simulations default to a 30-second interval.

In [9]:
async_request = main.StartSimulationRequest(
    vehicle_id="TRUCK-ASYNC", route_id="gps-demo",
    speed_kmh=45, interval_seconds=1, auto_start=True,
)
async_simulation = main.GPSSimulation(async_request, demo_route)
async_simulation.task = asyncio.create_task(main._run_gps_simulation(async_simulation))
await asyncio.sleep(1.1)
main._cancel_simulation_task(async_simulation)
await asyncio.sleep(0)
print("History records:", len(async_simulation.history))
print("Latest automatic update:", async_simulation.history[-1].model_dump(mode="json"))
print("Task reference cleared:", async_simulation.task is None)

History records: 2
Latest automatic update: {'lat': 30.733385268977692, 'lon': 76.7794852689777, 'altitude_m': 349.9744193066919, 'speed_kmh': 45.0, 'timestamp': '2026-10-01T10:38:17.718987Z'}
Task reference cleared: True


## 8. API endpoints
The route provider is mocked and route data is written to a temporary file. These cells exercise the same HTTP endpoints your teammates call.

In [10]:
class DemoRouteProvider:
    async def get_routes(self, request):
        return [{
            "distance_km": 12.5, "duration_seconds": 1080,
            "geometry": [point_a, point_b],
            "routing_profile": "driving", "profile_specific": False,
        }]


demo_directory = TemporaryDirectory()
demo_store = main.RouteStore(Path(demo_directory.name) / "routes.json")
provider_patch = patch.object(main, "route_provider", DemoRouteProvider())
store_patch = patch.object(main, "route_store", demo_store)
provider_patch.start()
store_patch.start()
main.simulations.clear()
client = TestClient(main.app)
print("Temporary cache:", demo_store.path)

Temporary cache: C:\Users\anura\AppData\Local\Temp\tmpzdrohu5i\routes.json


In [11]:
created = client.post("/api/v1/routes", json=route_input)
route_id = created.json()["routes"][0]["route_id"]
print("POST /api/v1/routes:", created.status_code)
pprint(created.json()["routes"][0])

POST /api/v1/routes: 201
{'alternative_index': 0,
 'created_at': '2026-10-01T10:38:17.834495Z',
 'destination': {'lat': 30.8333, 'lon': 76.8794},
 'distance_km': 12.5,
 'duration_seconds': 1080.0,
 'geometry': [{'altitude_m': 350.0, 'lat': 30.7333, 'lon': 76.7794},
              {'altitude_m': 320.0, 'lat': 30.8333, 'lon': 76.8794}],
 'origin': {'lat': 30.7333, 'lon': 76.7794},
 'profile_specific': False,
 'route_id': '2338ce93212b4e92becb662d5ded7bdb',
 'routing_profile': 'driving',
 'source': 'osrm',
 'vehicle': {'axle_count': 6,
             'gvw_kg': 18000.0,
             'height_m': 4.2,
             'truck_type': 'HCV',
             'width_m': 2.5},
 'vehicle_constraints_verified': False,
 'warnings': ['No dedicated HCV OSRM profile is configured; vehicle road '
              'restrictions are unverified.',
              'The API records vehicle dimensions but cannot verify that the '
              'OSRM profile enforces each individual limit.']}


In [12]:
listed = client.get("/api/v1/routes")
fetched = client.get(f"/api/v1/routes/{route_id}")
print("GET /api/v1/routes:", listed.status_code, "count:", len(listed.json()))
print("GET /api/v1/routes/{route_id}:", fetched.status_code)
print("Fetched geometry points:", len(fetched.json()["geometry"]))

GET /api/v1/routes: 200 count: 1
GET /api/v1/routes/{route_id}: 200
Fetched geometry points: 2


In [13]:
started = client.post("/api/v1/simulations", json={
    "vehicle_id": "TRUCK-API", "route_id": route_id,
    "speed_kmh": 45, "interval_seconds": 30, "auto_start": False,
})
print("POST /api/v1/simulations:", started.status_code)
pprint(started.json())
print("GET current:", client.get("/api/v1/simulations/TRUCK-API").json())
print("GET history:", client.get("/api/v1/simulations/TRUCK-API/history").json())

POST /api/v1/simulations: 201
{'altitude_m': 350.0,
 'lat': 30.7333,
 'lon': 76.7794,
 'route_id': '2338ce93212b4e92becb662d5ded7bdb',
 'route_progress_percent': 0.0,
 'speed_kmh': 45.0,
 'status': 'PAUSED',
 'timestamp': '2026-10-01T10:38:17.844899Z',
 'vehicle_id': 'TRUCK-API'}
GET current: {'lat': 30.7333, 'lon': 76.7794, 'altitude_m': 350.0, 'speed_kmh': 45.0, 'timestamp': '2026-10-01T10:38:17.844899Z', 'vehicle_id': 'TRUCK-API', 'route_id': '2338ce93212b4e92becb662d5ded7bdb', 'status': 'PAUSED', 'route_progress_percent': 0.0}
GET history: [{'lat': 30.7333, 'lon': 76.7794, 'altitude_m': 350.0, 'speed_kmh': 45.0, 'timestamp': '2026-10-01T10:38:17.844899Z'}]


In [14]:
ticked = client.post(
    "/api/v1/simulations/TRUCK-API/tick",
    json={"advance_seconds": 30},
)
print("POST tick:", ticked.status_code)
pprint(ticked.json())
print("History records:", len(client.get(
    "/api/v1/simulations/TRUCK-API/history"
).json()))

POST tick: 200
{'altitude_m': 349.2325792007575,
 'lat': 30.735858069330806,
 'lon': 76.7819580693308,
 'route_id': '2338ce93212b4e92becb662d5ded7bdb',
 'route_progress_percent': 2.5581,
 'speed_kmh': 45.0,
 'status': 'PAUSED',
 'timestamp': '2026-10-01T10:38:47.844899Z',
 'vehicle_id': 'TRUCK-API'}
History records: 2


In [15]:
print("pause:", client.post("/api/v1/simulations/TRUCK-API/pause").json()["status"])
print("resume:", client.post("/api/v1/simulations/TRUCK-API/resume").json()["status"])
print("pause again:", client.post("/api/v1/simulations/TRUCK-API/pause").json()["status"])
print("stop:", client.post("/api/v1/simulations/TRUCK-API/stop").json()["status"])

pause: PAUSED
resume: RUNNING
pause again: PAUSED
stop: STOPPED


In [16]:
client.post("/api/v1/simulations", json={
    "vehicle_id": "TRUCK-FINISH", "route_id": route_id,
    "speed_kmh": 10000, "auto_start": False,
})
finished = client.post(
    "/api/v1/simulations/TRUCK-FINISH/tick",
    json={"advance_seconds": 3600},
)
print("Completion status:", finished.json()["status"])
print("Final coordinate:", finished.json()["lat"], finished.json()["lon"])

Completion status: COMPLETED
Final coordinate: 30.8333 76.8794


## 9. Error responses
Invalid coordinates are rejected by request validation; an unknown saved route returns 404 when starting a simulation.

In [17]:
bad_response = client.post("/api/v1/routes", json={
    "origin": {"lat": 120, "lon": 76},
    "destination": {"lat": 28, "lon": 77},
})
missing_route = client.post("/api/v1/simulations", json={
    "vehicle_id": "TRUCK-MISSING", "route_id": "no-such-route",
    "speed_kmh": 40,
})
print("Invalid coordinate status:", bad_response.status_code)
print("Unknown route:", missing_route.status_code, missing_route.json())

Invalid coordinate status: 422
Unknown route: 404 {'detail': 'Route not found'}


## 10. Optional live OSRM request
This is the only section that makes a real network request. Set OSRM_BASE_URL in the notebook kernel first. An India-wide route needs an OSRM instance loaded with an India road extract.

In [18]:
live_osrm_url = os.getenv("OSRM_BASE_URL")
if not live_osrm_url:
    print("Skipped: set OSRM_BASE_URL in the notebook kernel, then rerun.")
else:
    live_routes = await main.OSRMClient().get_routes(route_request)
    print(f"Returned {len(live_routes)} route(s) from {live_osrm_url}")
    pprint(live_routes[0])

Skipped: set OSRM_BASE_URL in the notebook kernel, then rerun.


## 11. Cleanup
Run this after the walkthrough to close the client and restore the original module objects.

In [19]:
client.close()
provider_patch.stop()
store_patch.stop()
main.simulations.clear()
demo_directory.cleanup()
print("Demo state cleaned up.")

Demo state cleaned up.
